# Wingbeat Band Spectral Peak Visualisation

Instead of pyin, we directly find the **dominant frequency in the 200–1000 Hz band**
from the precomputed log-mel features. This is fast, needs no raw audio, and has no
voicing model that can fail on noisy clips.

For each clip we:
1. Average the log-mel feature over the time axis → mean spectrum (64 values)
2. Restrict to the mel bins covering 200–1000 Hz
3. Take `argmax` → peak frequency in Hz

**Figures:**
1. Heatmap — median peak Hz per (species, domain)
2. Violin plots — peak Hz distribution per species × domain
3. Mean spectrum — all 64 mel bins per species, D5 vs field
4. Domain shift diagnostic — peak Hz distribution per domain (collapsed across species)

In [ ]:
import pickle
from collections import defaultdict
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Mel filterbank parameters (from resolved config) ────────────────────
SR      = 8000
N_MELS  = 64
FMIN_MEL = 0      # mel filterbank lower edge
FMAX_MEL = 4000   # mel filterbank upper edge

# Wingbeat band of interest
WINGBEAT_LO = 200   # Hz
WINGBEAT_HI = 1000  # Hz

FEATURE_PKL = Path("Development_data/feature/training_features.pkl")

SPECIES_NAMES = [
    "Ae. aegypti", "Ae. albopictus", "Cx. quinquefasciatus",
    "An. gambiae", "An. arabiensis", "An. dirus",
    "Cx. pipiens", "An. minimus", "An. stephensi",
]
DOMAIN_NAMES = ["D1", "D2", "D3", "D4", "D5"]
D_COLORS = {
    "D1": "#e41a1c", "D2": "#377eb8", "D3": "#4daf4a",
    "D4": "#984ea3", "D5": "#ff7f00",
}

# Compute centre frequency of each mel bin
mel_freqs = librosa.mel_frequencies(n_mels=N_MELS, fmin=FMIN_MEL, fmax=FMAX_MEL)
band_mask = (mel_freqs >= WINGBEAT_LO) & (mel_freqs <= WINGBEAT_HI)
band_idx  = np.where(band_mask)[0]

print(f"Mel bins covering {WINGBEAT_LO}–{WINGBEAT_HI} Hz: "
      f"bins {band_idx[0]}–{band_idx[-1]} "
      f"({len(band_idx)} bins, {mel_freqs[band_idx[0]]:.0f}–{mel_freqs[band_idx[-1]]:.0f} Hz)")

In [ ]:
print(f"Loading {FEATURE_PKL} …")
with open(FEATURE_PKL, "rb") as fh:
    payload = pickle.load(fh)
items = payload["items"]
print(f"  {len(items)} clips loaded")

# Per-clip stats
peak_hz   = defaultdict(list)   # (species, domain) -> [peak_hz, ...]
mel_sums  = defaultdict(lambda: np.zeros(N_MELS))
mel_counts = defaultdict(int)

for item in items:
    s = item["species_label"]
    d = item["domain_label"]
    feat = item["feature"].astype(np.float32)   # (time_frames, n_mels)
    mean_spec = np.mean(feat, axis=0)            # (n_mels,)

    # Peak in wingbeat band
    band_spec  = mean_spec[band_idx]
    peak_bin   = band_idx[np.argmax(band_spec)]
    peak_hz[(s, d)].append(float(mel_freqs[peak_bin]))

    # Accumulate mean spectra
    mel_sums[(s, d)]   += mean_spec
    mel_counts[(s, d)] += 1

mean_mel = {k: mel_sums[k] / mel_counts[k] for k in mel_sums}
print(f"  {len(peak_hz)} (species, domain) groups")
print()
print("Counts per group:")
for (s, d) in sorted(peak_hz):
    print(f"  {SPECIES_NAMES[s]:28s} {DOMAIN_NAMES[d]}  {len(peak_hz[(s,d)])} clips")

## Figures

In [ ]:
# ── Figure 1: Median peak Hz heatmap ─────────────────────────────────────
grid = np.full((len(SPECIES_NAMES), len(DOMAIN_NAMES)), np.nan)
for (s, d), vals in peak_hz.items():
    grid[s, d] = np.median(vals)

fig, ax = plt.subplots(figsize=(9, 6))
masked = np.ma.masked_invalid(grid)
im = ax.imshow(masked, aspect="auto", cmap="plasma", vmin=200, vmax=1000)
plt.colorbar(im, ax=ax, label="Median dominant frequency (Hz)")
ax.set_xticks(range(len(DOMAIN_NAMES)));  ax.set_xticklabels(DOMAIN_NAMES)
ax.set_yticks(range(len(SPECIES_NAMES))); ax.set_yticklabels(SPECIES_NAMES, fontsize=8)
ax.set_title("Median Dominant Wingbeat Frequency per (Species, Domain)\n"
             "grey = no data   |   if domain-invariant, each row should be one colour",
             fontweight="bold")
for s in range(len(SPECIES_NAMES)):
    for d in range(len(DOMAIN_NAMES)):
        if not np.isnan(grid[s, d]):
            ax.text(d, s, f"{grid[s,d]:.0f}", ha="center", va="center",
                    fontsize=7, color="white", fontweight="bold")
plt.tight_layout()
plt.savefig("spectral_peak_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → spectral_peak_heatmap.png")

In [ ]:
# ── Figure 2: Violin plots per species × domain ───────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle("Dominant Wingbeat Frequency Distribution per Species × Domain",
             fontsize=13, fontweight="bold")

for sp, ax in enumerate(axes.flat):
    ax.set_title(SPECIES_NAMES[sp], fontsize=9)
    data, labels, cols = [], [], []
    for d, dname in enumerate(DOMAIN_NAMES):
        vals = peak_hz.get((sp, d), [])
        if len(vals) >= 3:
            data.append(vals); labels.append(dname); cols.append(D_COLORS[dname])
    if data:
        vp = ax.violinplot(data, showmedians=True, showextrema=True)
        for body, col in zip(vp["bodies"], cols):
            body.set_facecolor(col); body.set_alpha(0.7)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylabel("Peak Hz", fontsize=8)
        ax.set_ylim(WINGBEAT_LO, WINGBEAT_HI)
    else:
        ax.text(0.5, 0.5, "no data", ha="center", va="center",
                transform=ax.transAxes, fontsize=8, color="grey")

plt.tight_layout()
plt.savefig("spectral_peak_violin.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → spectral_peak_violin.png")

In [ ]:
# ── Figure 3: Mean log-mel spectrum per species, D5 vs field ──────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle("Mean Log-Mel Spectrum per Species\nOrange = D5 (lab), others = field domains",
             fontsize=12, fontweight="bold")
bins = np.arange(N_MELS)

# Shade wingbeat band
for sp, ax in enumerate(axes.flat):
    ax.axvspan(band_idx[0], band_idx[-1], color="yellow", alpha=0.12, label="wingbeat band")
    ax.set_title(SPECIES_NAMES[sp], fontsize=9)
    plotted = False
    for d, dname in enumerate(["D1", "D2", "D3", "D4"]):
        if (sp, d) in mean_mel:
            ax.plot(bins, mean_mel[(sp, d)], color=D_COLORS[dname],
                    alpha=0.85, lw=1.3, label=dname)
            plotted = True
    if (sp, 4) in mean_mel:
        ax.plot(bins, mean_mel[(sp, 4)], color=D_COLORS["D5"],
                lw=2.2, label="D5", zorder=5)
        plotted = True
    if plotted:
        ax.set_xlabel("Mel bin", fontsize=7); ax.set_ylabel("Mean log energy", fontsize=7)
        ax.legend(fontsize=6, loc="upper right")
    else:
        ax.text(0.5, 0.5, "no data", ha="center", va="center",
                transform=ax.transAxes, fontsize=8, color="grey")

plt.tight_layout()
plt.savefig("spectral_peak_mean_spectra.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → spectral_peak_mean_spectra.png")

In [ ]:
# ── Figure 4: Domain shift diagnostic ────────────────────────────────────
# Histogram of peak frequencies per domain, collapsed across all species.
# If domain shift is real, D5 and field domains should have different distributions.
domain_peaks = defaultdict(list)
for (s, d), vals in peak_hz.items():
    domain_peaks[d].extend(vals)

fig, axes = plt.subplots(1, len(DOMAIN_NAMES), figsize=(15, 4), sharey=True)
fig.suptitle("Peak Frequency Distribution per Domain (all species pooled)\n"
             "If domain-invariant features matter, D5 and field domains should overlap",
             fontsize=11, fontweight="bold")
bins_hist = np.linspace(WINGBEAT_LO, WINGBEAT_HI, 25)

for d, (ax, dname) in enumerate(zip(axes, DOMAIN_NAMES)):
    vals = domain_peaks.get(d, [])
    if vals:
        ax.hist(vals, bins=bins_hist, color=D_COLORS[dname], alpha=0.8, edgecolor="white")
        ax.axvline(np.median(vals), color="black", lw=1.5, ls="--",
                   label=f"median={np.median(vals):.0f} Hz")
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "no data", ha="center", va="center",
                transform=ax.transAxes, color="grey")
    ax.set_title(dname, fontweight="bold")
    ax.set_xlabel("Peak Hz", fontsize=8)
    ax.set_xlim(WINGBEAT_LO, WINGBEAT_HI)
axes[0].set_ylabel("Clip count")

plt.tight_layout()
plt.savefig("spectral_peak_domain_shift.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → spectral_peak_domain_shift.png")

## How to read the results

- **Heatmap row is one colour** → peak frequency is consistent across domains for that species → domain-invariant signal, good candidate for auxiliary loss or explicit feature
- **Heatmap row varies by domain** → spectral peak shifts with domain → domain shift is real in the frequency dimension
- **Figure 4 D5 vs field overlap** → if distributions are similar, the spectral content is domain-invariant; if D5 peaks cluster at different Hz than field domains, that's the source of domain shift